<a href="https://colab.research.google.com/github/AntonioLLuis/Algoritmo-e-extrutura-de-dados/blob/main/forca_etapa_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# forca_comentado.py
# Jogo da Forca — versão completa com comentários explicativos linha por linha.

# importações de módulos da biblioteca padrão
import random            # fornece funções para operações aleatórias (ex.: choice)
import unicodedata       # utilitários Unicode (usado para normalização / remoção de acentos)

# HANGMAN_STAGES: lista de strings multilinha representando os estágios do desenho da forca.
# Cada elemento é uma 'fase' do boneco — quanto mais erros, maior o índice mostrado.
HANGMAN_STAGES = [
    """
     ------
     |    |
     |
     |
     |
     |
    _|_
    """,
    """
     ------
     |    |
     |    O
     |
     |
     |
    _|_
    """,
    """
     ------
     |    |
     |    O
     |    |
     |
     |
    _|_
    """,
    """
     ------
     |    |
     |    O
     |   /|
     |
     |
    _|_
    """,
    """
     ------
     |    |
     |    O
     |   /|\\
     |
     |
    _|_
    """,
    """
     ------
     |    |
     |    O
     |   /|\\
     |   /
     |
    _|_
    """,
    """
     ------
     |    |
     |    O
     |   /|\\
     |   / \\
     |
    _|_
    """
]

# Lista exemplo de palavras. Inicialmente sem acentuação para testes simples.
WORD_LIST = [
    "python", "computador", "programacao", "algoritmo",
    "desenvolvedor", "teclado", "mouse", "internet",
    "escola", "universidade", "projeto", "rede",
    "senha", "arquivo", "janela", "processador"
]

# -----------------------
# Função: remove_accents
# -----------------------
def remove_accents(s: str) -> str:
    """
    Recebe uma string 's' e retorna uma versão sem marcas diacríticas (acentos)
    e em minúsculas. As anotações ': str' e '-> str' são apenas hints de tipo.
    """
    nfkd = unicodedata.normalize('NFD', s)
    # unicodedata.normalize('NFD', s) decompõe caracteres acentuados em partes.
    # Ex.: 'á' -> 'a' + '´' (letra + marca). Guardamos o resultado em 'nfkd'.

    # Construímos uma nova string incluindo apenas caracteres cuja categoria Unicode
    # não seja 'Mn' (Mark, Nonspacing) — ou seja, removemos as marcas diacríticas.
    sem_acentos = ''.join(ch for ch in nfkd if unicodedata.category(ch) != 'Mn')

    # .lower() converte para minúsculas — útil para comparações case-insensitive.
    return sem_acentos.lower()

# -----------------------
# Função: escolher_palavra
# -----------------------
def escolher_palavra(palavras):
    """
    Retorna uma palavra aleatória da lista 'palavras'.
    Usa random.choice para selecionar um elemento aleatório de uma sequência.
    """
    return random.choice(palavras)

# -----------------------
# Função: mostrar_estado
# -----------------------
def mostrar_estado(palavra_original, letras_reveladas, tentativas_erradas, letras_chutadas):
    """
    Exibe:
    - o estágio atual do desenho da forca (HANGMAN_STAGES),
    - a palavra com letras reveladas e underscores para letras ocultas,
    - contagem de erros e lista de letras chutadas.
    """

    # min(tentativas_erradas, len(HANGMAN_STAGES)-1)
    # garante que o índice não ultrapasse o tamanho da lista HANGMAN_STAGES.
    indice = min(tentativas_erradas, len(HANGMAN_STAGES) - 1)
    print(HANGMAN_STAGES[indice])  # imprime o desenho apropriado

    # vamos montar a exibição da palavra posição a posição
    exibicao = []  # lista temporária para juntar caracteres/underscores
    for i, ch in enumerate(palavra_original):
        # enumerate cria pares (índice, caractere) para cada caractere da palavra
        if not ch.isalpha():
            # se o caractere não é uma letra (por exemplo espaço ou hífen),
            # mostramos ele diretamente — útil para frases ou palavras compostas.
            exibicao.append(ch)
        elif i in letras_reveladas:
            # se o índice foi revelado (o jogador já descobriu essa posição),
            # mostramos a letra original (com acento, maiúscula etc).
            exibicao.append(ch)
        else:
            # caso contrário, mostramos underscore para indicar letra oculta.
            exibicao.append("_")

    # juntamos a lista 'exibicao' com espaços entre caracteres para melhor leitura
    print("Palavra: ", " ".join(exibicao))

    # mostramos quantidade de erros (tentativas_erradas) e o máximo possível
    print(f"Erros: {tentativas_erradas} / {len(HANGMAN_STAGES)-1}")

    # mostramos as letras chutadas ordenadas alfabeticamente
    # sorted(letras_chutadas) retorna uma lista ordenada; " ".join(...) transforma em string
    print("Letras chutadas:", " ".join(sorted(letras_chutadas)))
    print()  # imprime linha em branco para separação visual

# -----------------------
# Função principal: jogo_forca
# -----------------------
def jogo_forca(palavras, max_erros=None):
    """
    Controla o fluxo principal do jogo.
    'palavras' é a lista de palavras possíveis.
    'max_erros' se fornecido substitui o padrão (len(HANGMAN_STAGES)-1).
    """
    # escolhe a palavra que será usada nesta partida
    palavra_original = escolher_palavra(palavras)

    # palavra_norm é a versão sem acentos utilizada apenas para comparações
    palavra_norm = remove_accents(palavra_original)

    # se max_erros não foi fornecido, usamos o padrão (número de estágios - 1)
    if max_erros is None:
        max_erros = len(HANGMAN_STAGES) - 1

    # estruturas de suporte:
    letras_reveladas = set()   # set de índices (posições) cuja letra já foi revelada
    letras_chutadas = set()    # set de letras (normalizadas) que o jogador já tentou
    erros = 0                  # contador de erros cometidos

    # mapeamento letra_normalizada -> lista de índices onde aparece na palavra_original
    posicoes_por_letra = {}
    for i, ch in enumerate(palavra_original):
        if ch.isalpha():
            norm = remove_accents(ch)  # normaliza a letra (remove acento)
            # setdefault cria a lista se a chave não existir e retorna a lista
            posicoes_por_letra.setdefault(norm, []).append(i)

    # instruções iniciais para o jogador
    print("Jogo da Forca - digite 'desistir' para abandonar")
    mostrar_estado(palavra_original, letras_reveladas, erros, letras_chutadas)

    # loop principal do jogo — repete até break (vitória/derrota/desistir)
    while True:
        # solicita o chute do jogador
        chute = input("Seu chute (letra / palavra / desistir): ").strip()
        # .strip() remove espaços no início/fim da entrada

        if not chute:
            # se a string está vazia (usuário só apertou Enter), pedimos algo válido
            print("Digite algo.")
            continue  # volta para o início do loop sem processar mais

        # tratamento da opção 'desistir' (aceitamos maiúsculas/minúsculas)
        if chute.lower() == "desistir":
            # .lower() converte para minúsculas antes da comparação
            print("Você desistiu. Palavra:", palavra_original)
            break  # encerra o loop principal (e a função termina)

        # normalizamos o chute para comparar sem acentos/maiúsculas
        chute_norm = remove_accents(chute)

        # se o chute tem mais de um caractere, consideramos tentativa de palavra inteira
        if len(chute_norm) > 1:
            # comparamos a versão normalizada do chute com a palavra normalizada
            if chute_norm == remove_accents(palavra_original):
                print("Parabéns! Você acertou a palavra:", palavra_original)
                break
            else:
                # chute de palavra errado conta como um erro
                erros += 1
                print("Palavra incorreta! (+1 erro)")
                mostrar_estado(palavra_original, letras_reveladas, erros, letras_chutadas)
        else:
            # chute de uma única letra — validamos e processamos
            letra = chute_norm
            # validamos: letra deve ser alfabética e ter comprimento 1
            if not letra.isalpha() or len(letra) != 1:
                print("Digite apenas letras.")
                continue

            # já chutou essa letra antes?
            if letra in letras_chutadas:
                print("Você já chutou essa letra:", letra)
                continue

            # registra a letra nas chutadas
            letras_chutadas.add(letra)

            # verifica se a letra normalizada está no mapeamento de posições
            if letra in posicoes_por_letra:
                # revela todas as posições onde essa letra ocorre
                for pos in posicoes_por_letra[letra]:
                    letras_reveladas.add(pos)
                print("Boa! Letra correta.")
                mostrar_estado(palavra_original, letras_reveladas, erros, letras_chutadas)
            else:
                # letra não existe => conta como erro
                erros += 1
                print("Letra errada! (+1 erro)")
                mostrar_estado(palavra_original, letras_reveladas, erros, letras_chutadas)

        # checagem de vitória: todas as posições de letras foram reveladas?
        todas_posicoes_letras = {i for i, ch in enumerate(palavra_original) if ch.isalpha()}
        # set comprehension acima cria conjunto com índices de todas as letras da palavra

        if todas_posicoes_letras.issubset(letras_reveladas):
            # issubset retorna True se todas_posicoes_letras ⊆ letras_reveladas
            print("Parabéns — você revelou todas as letras! Palavra:", palavra_original)
            break

        # checagem de derrota: número de erros alcançou o máximo permitido?
        if erros >= max_erros:
            print("Você perdeu! A palavra era:", palavra_original)
            break

# -----------------------
# Bloco principal — executa quando o arquivo é rodado diretamente
# -----------------------
if __name__ == "__main__":
    # if __name__ == "__main__": evita que esse bloco rode se o arquivo for importado como módulo

    # pergunta ao usuário se quer usar sua própria lista de palavras
    usar_lista = input("Deseja usar sua própria lista de palavras? (s/n): ").strip().lower()

    if usar_lista == "s":
        # coleta palavras do usuário até ele digitar uma linha em branco
        palavras = []
        print("Digite palavras (enter em branco para finalizar):")
        while True:
            p = input("Palavra: ").strip()
            if not p:  # se o usuário apenas apertou Enter, encerra a coleta
                break
            palavras.append(p)
        # se o usuário não digitou nenhuma palavra, usamos a lista padrão
        if palavras:
            WORDS = palavras
        else:
            WORDS = WORD_LIST
    else:
        # usuário não quer fornecer lista própria -> usamos a lista padrão
        WORDS = WORD_LIST

    # pergunta opcional para alterar o número máximo de erros
    entrada = input(f"Máx de erros (pressione Enter para {len(HANGMAN_STAGES)-1}): ").strip()
    try:
        # tenta converter para inteiro. se entrada vazia -> max_err = None -> usa padrão
        max_err = int(entrada) if entrada else None
    except ValueError:
        # tratamento caso o usuário digite texto que não pode virar inteiro
        print("Entrada inválida; será usado o padrão.")
        max_err = None

    # chama a função principal com a lista final de palavras e max_err (pode ser None)
    jogo_forca(WORDS, max_err)


Deseja usar sua própria lista de palavras? (s/n): n
Máx de erros (pressione Enter para 6): d
Entrada inválida; será usado o padrão.
Jogo da Forca - digite 'desistir' para abandonar

     ------
     |    |
     |
     |
     |
     |
    _|_
    
Palavra:  _ _ _ _ _ _ _ _ _
Erros: 0 / 6
Letras chutadas: 

Seu chute (letra / palavra / desistir): k
Letra errada! (+1 erro)

     ------
     |    |
     |    O
     |
     |
     |
    _|_
    
Palavra:  _ _ _ _ _ _ _ _ _
Erros: 1 / 6
Letras chutadas: k

Seu chute (letra / palavra / desistir): w
Letra errada! (+1 erro)

     ------
     |    |
     |    O
     |    |
     |
     |
    _|_
    
Palavra:  _ _ _ _ _ _ _ _ _
Erros: 2 / 6
Letras chutadas: k w



KeyboardInterrupt: Interrupted by user